# RSA / BBQ Fashion Search Demo

A real interactive fashion search app with product images.

The demo uses a strict held-out split of Fashion Product Images Small and compares the same finalist methods used in the experiments:

- **BBQ1 + int4** — 56 B/item, ~216 B/semantic predicate
- **PQ64** — 64 B/item, ~65 KB/predicate
- **RSA2** — 96 B/item, sparse learned LUT program
- **FP32 linear** — full-precision semantic head
- **Dense MiniLM** — retrieval baseline

Try queries such as **`minimalist black office shoes not sporty`**. Exact catalog constraints stay exact; latent terms are executed as learned semantic predicates. The image captions optionally show whether the independent CLIP image teacher considers each returned item a match.


In [ ]:
#@title 1) Clone repo and install demo dependencies
import os, pathlib, shutil, subprocess
ROOT = pathlib.Path('/content/ras')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run(['pip','install','-q','-e','.','faiss-cpu','gradio'], check=True)
print('repo:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())


In [ ]:
#@title 2) Check runtime
import torch
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('Tip: Runtime → Change runtime type → GPU. The demo still works on CPU, but CLIP preparation is slower.')


In [ ]:
#@title 3) Prepare held-out fashion catalog and semantic programs
# First run encodes 8k images/titles and fits the four demo methods.
# Subsequent reruns in the same runtime reuse the representation cache.
import os, time
os.chdir('/content/ras')
from demos.fashion_app import prepare_demo
t0 = time.time()
state = prepare_demo('configs/binary_bbq_smoke.yaml')
print(f'ready in {(time.time()-t0)/60:.1f} minutes')
print(f'visible held-out products: {len(state.df_test):,}')


In [ ]:
#@title 4) Launch fashion app
from demos.fashion_app import build_app
app = build_app(state)
app.launch(share=True, debug=False)
